In [137]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [138]:
import seaborn as sns

df = sns.load_dataset("penguins")
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [139]:
df = df.dropna()

In [140]:
y = df["species"]
X = df.drop(columns=["species"])

In [141]:
print(df["species"].unique())


['Adelie' 'Chinstrap' 'Gentoo']


# Creating Decision Tree

In [142]:

def getsplit(X, column):
    # we will make a list that contains every possible split for this column 
    values = X[column].unique()
    if values.dtype == "object":
        return values.tolist()
    else:
        values = sorted(values)
        split = []
        for i in range(len(values)-1):
            avg = (values[i]+values[i+1])/2
            split.append(round(avg, 2))
        return split


In [143]:
def split_data(df, column, value):
    # if it is an object we will split the data : the left will take the lines where col == value and right will take the rest
    if df[column].dtype == "object":
        left = df[df[column] == value]
        right = df[df[column] != value]
    # if it is an int/float we will split the data : the left will take the lines where col < value and right will take the rest
    else:
        left = df[df[column] < value]
        right = df[df[column] >= value]
    return (left,right)

In [144]:
def gini_impurity(data, target):
    count = {}
    for key in data[target]:
        if key not in count:
            count[key] = 1
        else :
            count[key] += 1
    total = len(data)
    gi = 1
    for key in count :
        probability = count[key] / total
        gi -= probability**2
    return gi
    

In [145]:
def weighted_gini(left, right, target):
    nb_left, nb_right = len(left), len(right)
    total = nb_left + nb_right
    total_gi = (nb_left / total) * gini_impurity(left, target) + (nb_right / total) * gini_impurity(right, target)
    return total_gi

In [146]:
def best_split(df, target, nb_features):
    # we intialize the variables
    best_wg, best_split, best_column = float("inf"), None, None
    X = df.drop(target, axis = 1)
    # choosing random features for random forest
    features = list(X.columns)
    if nb_features is None :
        nb_features = int(np.sqrt(len(features)))
    else :
        nb_features = min(nb_features, len(features))
    selected_features = np.random.choice(features, size=nb_features, replace=False)
    for column in selected_features:
        # for every column we will get a list of the possible splits
        splits = getsplit(X, column)
        for split in splits:
            # for every split we will split it again so that it would be left, right respecting the rules that we mentionned before
            left, right = split_data(df, column, split)
            if len(left) == 0 or len(right) == 0:
                continue
            wg = weighted_gini(left, right, target)
            # choosing the best split (the one that has the least wg)
            if wg < best_wg :
                best_wg = wg
                best_split = split
                best_column = column
    return best_wg, best_split, best_column

In [147]:
def should_stop(df, target, depth, max_depth, min_samples_split):
    if gini_impurity(df, target) == 0: # it is pure
        return True
    elif max_depth is not None and depth >= max_depth : # max depth reached
        return True
    elif len(df) < min_samples_split : # not enough samples
        return True
    return False

In [148]:
def create_leaf(df, target):
    # we create a leaf and the most appearing key will be the predicted value 
    count = {}
    # we count the number of times that a key appears
    for key in df[target]:
        if key not in count:
            count[key] = 1
        else :
            count[key] += 1
    max_apperance_nb = 0
    max_apperance_key = None
    # we take the one that appeared the most
    for key in count :
        if count[key]>max_apperance_nb:
           max_apperance_nb = count[key]
           max_apperance_key = key
    return {"type": "leaf" , "prediction" : max_apperance_key}
        

In [149]:
def build_tree(df, target, depth=0, max_depth, min_samples_split, nb_features):
    # reccursive function that build a tree
    if should_stop(df, target, depth, max_depth, min_samples_split):
        return create_leaf(df, target)
    best_wg, best_value, best_column = best_split(df, target, nb_features)
    if best_column is None:
        return create_leaf(df, target)
    left, right = split_data(df, best_column, best_value)
    left_tree = build_tree(left, target, depth+1, max_depth, min_samples_split)
    right_tree = build_tree(right, target, depth+1, max_depth, min_samples_split)
    return {
    "type": "node",
    "column": best_column,
    "value": best_value,
    "left": left_tree,
    "right": right_tree
    }

# Creating Random Forest

In [ ]:
def bootstrap(train):
    # we'll create a dataset that contains random lines from the training dataset 
    n = len(train)
    bootstrap_indices = np.random.choice(n, size=n, replace=True)
    used_indices = set(bootstrap_indices)
    all_indices = set(range(n))
    # we're trying to make a dataset from the all the lines that we didn't use called "out_of_bag"
    out_of_bag_indices = list(all_indices - used_indices)
    bootstrap_df = train.iloc[bootstrap_indices].reset_index(drop=True)
    out_of_bag_df = train.iloc[out_of_bag_indices].reset_index(drop=True)
    return bootstrap_df , out_of_bag_df
def creating_random_forest(train, target, n_trees=100, max_depth=None, min_samples_split=2, nb_features=None):
    # this function returns a list containing n decision trees and out_of_bag dataseats corresponding to those trees
    forest = []
    for _ in range(n_trees):
        bootstrap_df , out_of_bag_df = bootstrap(train)
        tree = build_tree(bootstrap_df, target, 0, max_depth, min_samples_split, nb_features)
        forest.append((tree,out_of_bag_df))
    return forest
    

# Predicting result

In [ ]:
def predict_one_sample(forest, sample):
    votes = {}
    for tree,_ in forest:
        label = predict_using_one_tree(tree, sample)
        if label not in prediction:
            votes[label] = 1
        else:
            votes[label] += 1
    prediction = max(votes, key=votes.get)
    return prediction

In [151]:
def predict_multiple_lines(forest, X):
    predictions = []
    for _, row in X.iterrows():
        predictions.append(predict_one_sample(forest, row))
    return predictions

In [150]:
def predict_using_one_tree(tree, sample):
    if tree["type"] == "leaf" :
        return tree["prediction"]
    column = tree["column"]
    value = tree["value"]

    if isinstance(value, (int, float, np.number)): # we see if the value is numerical or not
        if sample[column] < value :
            return predict(tree["left"], sample) 
        else:
            return predict(tree["right"], sample)
    else:
        if sample[column] == value :
            return predict(tree["left"], sample) 
        else:
            return predict(tree["right"], sample)
        
    

In [152]:
def accuracy(y_true, y_pred):
    correct = 0

    for true, pred in zip(y_true, y_pred):
        if true == pred:
            correct += 1

    return correct / len(y_true)

In [153]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

In [154]:
tree = build_tree(train, "species", max_depth=3)

In [155]:
X_test = test.drop("species", axis=1)
y_test_me = test["species"]

predictions = predict_multiple_lines(tree, X_test)

In [156]:
print(accuracy(y_test_me, predictions))

0.9850746268656716


# Creating the same model using sklearn

In [157]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X = df.drop("species", axis=1)
y = df["species"]
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X["island"] = encoder.fit_transform(X["island"])
X["sex"] = encoder.fit_transform(X["sex"])
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
model = DecisionTreeClassifier(
    max_depth=3
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# my model vs scikit-learn 

In [158]:
print(f" my accuracy :{accuracy(y_test_me, predictions)} vs scikit-learn's :{accuracy_score(y_test, y_pred)}")

 my accuracy :0.9850746268656716 vs scikit-learn's :0.9850746268656716


# Conclusion : it gives the same accuracy